# Cloud GPU Training Pipeline: Explainable AI for Early Diabetes Risk Screening

**Project**: Explainable AI for Early Diabetes Risk Screening Using Retinal Fundus Images in Rural India  
**Repository**: [https://github.com/jay45-cyber/Early-Diabetic_Retinopathi](https://github.com/jay45-cyber/Early-Diabetic_Retinopathi)  
**Target Environment**: Google Colab (Tesla T4 / A100 GPU Recommended)  

---

### Overview & Clinical Integrity Rules
This notebook provides a cloud training pipeline for training and evaluating deep neural networks (ResNet50, EfficientNetV2, ConvNeXt, RETFound) on retinal fundus photographs.

> [!IMPORTANT]
> 1. **Zero Data Leakage**: Bilateral eyes (OD/OS) and serial visits for every patient are strictly kept in the exact same partition.
> 2. **Decoupled Ground Truth**: Diabetic Retinopathy grades are NOT substitutes for laboratory-confirmed diabetes ground truth.
> 3. **Subgroup Analysis**: The primary clinical target is evaluating model discrimination on the **$	ext{DR} = 0$ subgroup (no visible retinopathy)**.
> 4. **No Secrets**: Never paste GitHub Personal Access Tokens or API keys directly into this notebook.

## 1. Verify GPU Hardware Acceleration

In [ ]:
import torch
import os
import sys

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"SUCCESS: Dedicated GPU detected: {gpu_name} ({gpu_mem:.2f} GB VRAM)")
    !nvidia-smi
else:
    print("WARNING: No GPU detected. Execution will proceed on CPU with reduced batch size.")

## 2. Clone Repository & Setup Environment

In [ ]:
# Clone authorized public repository if running in fresh Colab session
REPO_NAME = "Early-Diabetic_Retinopathi"
if not os.path.exists(REPO_NAME):
    !git clone https://github.com/jay45-cyber/Early-Diabetic_Retinopathi.git
    %cd Early-Diabetic_Retinopathi
else:
    %cd Early-Diabetic_Retinopathi
    !git pull origin main

# Install pinned dependencies
!pip install -q -r requirements.txt
!pip install -q pytest

# Verify test suite
!pytest tests/ -v

## 3. Mount Google Drive for Authorized Clinical Data & Large Checkpoints

Large clinical image archives and multi-gigabyte checkpoints should be stored on Google Drive to survive Colab session disconnections.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Designate Drive paths
DRIVE_DATA_DIR = "/content/drive/MyDrive/medical_ai_data/retina/raw"
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/medical_ai_checkpoints/diabetes_retina"
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)

print(f"Drive data path configured: {DRIVE_DATA_DIR}")
print(f"Drive checkpoint path configured: {DRIVE_CHECKPOINT_DIR}")

## 4. Run Dataset Ingestion & Leakage-Safe Splitting

In [ ]:
# Example: Ingest BRSET or custom clinical dataset
# !python scripts/build_splits.py --manifest data/manifests/patient_manifest.csv --output_dir data/manifests

from src.data.grouped_split import create_patient_grouped_splits, verify_zero_leakage
import pandas as pd

manifest_path = "data/manifests/patient_manifest.csv"
if os.path.exists(manifest_path):
    df = pd.read_csv(manifest_path)
    splits = create_patient_grouped_splits(df, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15)
    audit = verify_zero_leakage(splits["train"], splits["val"], splits["test"])
    print("Split verification passed:", audit)
else:
    print("Manifest not found. Connect authorized raw dataset to proceed.")

## 5. Controlled Baseline Model Training (ResNet50 / ConvNeXt)

Trains the baseline model with early stopping, weight decay, and validation AUROC tracking.

In [ ]:
from src.utils.seed import seed_everything
from src.models.baseline_resnet import BaselineResNet50
from src.models.convnext import RetinalConvNeXt

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Instantiate model
model = BaselineResNet50(pretrained=True, dropout_rate=0.3).to(device)
print(f"Model instantiated on {device}.")

# Training loop connects directly to GPU DataLoader when manifests are populated
# Checkpoints automatically saved to DRIVE_CHECKPOINT_DIR to allow training resumption.

## 6. Evaluation, Probability Calibration & Subgroup Analysis

In [ ]:
from src.evaluation.classification_metrics import compute_screening_metrics
from src.evaluation.calibration import PlattCalibrator, compute_expected_calibration_error
from src.evaluation.subgroup_analysis import evaluate_subgroups

print("Evaluation modules loaded. Metrics evaluated: AUROC, AUPRC, Brier Score, Platt Calibration, DR=0 Subgroup.")

## 7. Explainability Visualizations (Grad-CAM++)

Computes saliency heatmaps on the final convolutional layer.

In [ ]:
from src.explainability.gradcam import GradCAMPlusPlus

gradcam = GradCAMPlusPlus(model, target_layer=model.backbone.layer4[-1])
print("Grad-CAM++ hook registered successfully.")

## 8. Resuming Interrupted Training & Colab Limitations

- **Session Disconnection**: Colab free tier sessions may disconnect after inactivity or timeout (12 hours max).
- **Resume Pattern**: To resume training, load the last checkpoint saved on Google Drive:
  ```python
  checkpoint = torch.load(f"{DRIVE_CHECKPOINT_DIR}/baseline_resnet50_best.pt")
  model.load_state_dict(checkpoint["model_state_dict"])
  ```
- **Artifact Safety**: Evaluation metrics and figures are written to `outputs/`. Large checkpoints are saved to Google Drive and excluded from Git tracking via `.gitignore`.